### Integrated Analysis Report: Unharmonized, Harmonized, scETM, Scanorama, Harmony, and scVI

- **Author**: Automated report from `classifer.ipynb`
- **Date**: 2025-09-30

### Results (KMeans on embeddings; metrics vs. `perturbation`)

| Method                         | Embedding         | NMI    | ARI    |
|--------------------------------|-------------------|--------|--------|
| Unharmonized PCA               | `X_pca`           | 0.3170 | 0.1755 |
| Harmonized PCA                 | `X_pca`           | 0.3647 | 0.2108 |
| Scanorama (counts)             | `X_scanorama`     | 0.2359 | 0.1112 |
| Scanorama on scETM             | `X_scanorama`     | 0.3052 | 0.1898 |
| Harmony (PCA of counts)        | `X_pca_harmony`   | 0.3054 | 0.1657 |
| Harmony on scETM               | `X_pca_harmony`   | 0.3023 | 0.1692 |
| scVI                           | `X_scvi`          | 0.3199 | 0.2228 |

### Brief method notes
- We compute latent embeddings using standard tools: PCA, Scanorama, Harmony, scETM, and scVI.
- We run `KMeans(n_clusters=10)` on each embedding and score clusters with **NMI** and **ARI** against the `perturbation` labels.
- The integration utilities (e.g., `run_scanorama_integration`, `run_harmony_integration`, `run_scvi_integration`, `run_scetm_integration`) produce the respective embeddings stored under `obsm` (e.g., `X_scanorama`, `X_pca_harmony`, `X_scvi`).

### Observations
- Best NMI: **Harmonized PCA (0.3647)**.
- Best ARI: **scVI (0.2228)**.
- Using scETM as input benefits Scanorama/Harmony vs. counts, but still trails Harmonized PCA (NMI) and scVI (ARI).



In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.cluster import KMeans


warnings.filterwarnings('ignore')
np.random.seed(42)
sc.settings.verbosity = 3

# Instead of set_figure_params, configure matplotlib manually
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.dpi'] = 100

/home/krupavardhan4869@gmail.com/disenv/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
harmonized_data = sc.read_h5ad("downloaded_adata_diseases_harmonized.h5ad")

In [3]:
harmonized_data

AnnData object with n_obs × n_vars = 780000 × 5634
    obs: 'dataset_id', 'donor_id', 'tissue_general', 'perturbation', 'chunk_number', 'filename', 'cell_index', 'dataset_dir', 'cell_type_original', 'filepath', 'filepath_harmonized', 'state', 'cell_type_level1', 'total_counts', 'n_nonzero_genes', 'max_count'

In [4]:
parse = sc.read_h5ad("adata_parse.h5ad")

In [7]:
parse = parse[:780000]

In [8]:
parse.var_names = [str(i) for i in range(parse.n_vars)]
harmonized_data.var_names = [str(i) for i in range(harmonized_data.n_vars)]

In [9]:
harmonized_data.obs['batch'] = 'batch1'
parse.obs['batch'] = 'batch2'

In [10]:
combined_har_data = sc.concat([harmonized_data,parse], axis=0,join='inner')

In [11]:
combined_har_data

AnnData object with n_obs × n_vars = 1560000 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch'

In [12]:
print("normalized the data for harmonized data")
sc.pp.normalize_total(combined_har_data, target_sum=1e4)
sc.pp.log1p(combined_har_data)
sc.tl.pca(combined_har_data, n_comps=50)

normalized the data for harmonized data
normalizing counts per cell
    finished (0:00:02)
computing PCA
    with n_comps=50
    finished (0:06:44)


In [17]:
combined_har_data.write_h5ad("classifer/combined_har_data.h5ad")

In [19]:
combined_har_data

AnnData object with n_obs × n_vars = 1560000 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [20]:
combined_har_data = sc.read_h5ad("classifer/combined_har_data.h5ad")

In [21]:
combined_har_data

AnnData object with n_obs × n_vars = 780000 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [22]:
combined_data_scetm = sc.read_h5ad("classifer/combined_unharm_scetm.h5ad")

In [23]:
combined_data_scetm

AnnData object with n_obs × n_vars = 1560000 × 50
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types', '_scvi_batch', '_scvi_labels'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'log1p', 'pca'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scanorama', 'X_scvi'

In [8]:
combined_har_data = sc.read_h5ad("classifer/combined_har_data.h5ad")
combined_har_data

AnnData object with n_obs × n_vars = 218713 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch'
    uns: 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'

In [7]:
combined_har_data = combined_har_data[:780000]
combined_har_data.write_h5ad("classifer/combined_har_data.h5ad")

In [237]:
from models.harmony import run_harmony_integration
from models.scanorama import run_scanorama_integration
from models.scVI_model import run_scvi_integration
from models.scETM import run_scetm_integration

In [238]:
%matplotlib inline

In [239]:
unharmonized_data = sc.read_h5ad("downloaded_adata_diseases_unharmonized.h5ad")
harmonized_data = sc.read_h5ad("downloaded_adata_diseases_harmonized.h5ad")

In [258]:
unharmonized_data.obs['perturbation'].value_counts()


perturbation
B-cell acute lymphoblastic leukemia           78000
COVID-19                                      78000
hematologic disorder                          78000
juvenile dermatomyositis                      78000
myelodysplastic syndrome                      78000
myelodysplastic/myeloproliferative disease    78000
normal                                        78000
post-COVID-19 disorder                        78000
respiratory system disorder                   78000
systemic lupus erythematosus                  78000
Name: count, dtype: int64

In [240]:
parse = sc.read_h5ad("adata_parse.h5ad")

In [241]:
group_key = "perturbation"
n_cells = 78000
output_file = "adata_downsampled.h5ad"

In [242]:
sampled_indices = []

for group in harmonized_data.obs[group_key].unique():
    group_indices = np.where(harmonized_data.obs[group_key] == group)[0]
    
    if len(group_indices) < n_cells:
        raise ValueError(f"Group {group} has only {len(group_indices)} cells, less than requested {n_cells}")
    
    sampled = np.random.choice(group_indices, n_cells, replace=False)
    sampled_indices.extend(sampled)

In [243]:
adata_downsampled_unhar = harmonized_data[sampled_indices].copy()


In [244]:
sampled_indices = []

for group in unharmonized_data.obs[group_key].unique():
    group_indices = np.where(unharmonized_data.obs[group_key] == group)[0]
    
    if len(group_indices) < n_cells:
        raise ValueError(f"Group {group} has only {len(group_indices)} cells, less than requested {n_cells}")
    
    sampled = np.random.choice(group_indices, n_cells, replace=False)
    sampled_indices.extend(sampled)

In [245]:
adata_downsampled_har = unharmonized_data[sampled_indices].copy()

In [246]:
true_labels = adata_downsampled_unhar.obs['perturbation'].values


In [247]:
true_labels.size

780000

In [248]:
adata_downsampled_unhar.obs['batch'] ='batch1'
adata_downsampled_har.obs['batch'] ='batch1'

In [249]:
sample_size = 780000
parse = parse[:sample_size]
parse.shape
parse.obs['batch'] ='batch2'

In [250]:
adata_downsampled_unhar.var_names = [str(i) for i in range(adata_downsampled_unhar.n_vars)]
parse.var_names = [str(i) for i in range(parse.n_vars)]
adata_downsampled_har.var_names = [str(i) for i in range(adata_downsampled_har.n_vars)]

In [251]:
combined_data = sc.concat([adata_downsampled_unhar,parse], axis=0,join='inner')
combined_har_data = sc.concat([adata_downsampled_har,parse], axis=0,join='inner')

In [252]:
combined_data

AnnData object with n_obs × n_vars = 1560000 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch'

In [253]:
sc.pp.normalize_total(adata_downsampled_unhar, target_sum=1e4)
sc.pp.log1p(adata_downsampled_unhar)
sc.tl.pca(adata_downsampled_unhar, n_comps=50)

normalizing counts per cell


    finished (0:00:00)
computing PCA
    with n_comps=50
    finished (0:02:30)


In [254]:
def kmeans_clustering(adata,n_clusters =10,embeddings='X_pca'):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    adata.obs['cluster'] = kmeans.fit_predict(adata.obsm[embeddings])
    return adata.obs['cluster'].values


In [255]:
kmeans_clustering(adata_downsampled_unhar)

array([2, 4, 3, ..., 6, 0, 0], shape=(780000,), dtype=int32)

In [256]:
pred_labels = adata_downsampled_unhar.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.3183
ARI score: 0.1747


In [259]:
sc.pp.normalize_total(adata_downsampled_har, target_sum=1e4)
sc.pp.log1p(adata_downsampled_har)
sc.tl.pca(adata_downsampled_har, n_comps=50)

normalizing counts per cell
    finished (0:00:00)
computing PCA
    with n_comps=50
    finished (0:01:28)


In [260]:
kmeans_clustering(adata_downsampled_har)

array([2, 5, 0, ..., 3, 5, 5], shape=(780000,), dtype=int32)

In [261]:
pred_labels = adata_downsampled_har.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.4273
ARI score: 0.2802


In [263]:
combined_data.write_h5ad("combined_data_with_parse.h5ad")

In [264]:
run_scetm_integration(combined_data,dataset_key='batch',celltype_key ='cell_type_level1')   

[2025-09-30 12:00:43,381] INFO - scETM.logging_utils: scETM.__init__(5634, 2, enable_batch_bias = True)
[2025-09-30 12:00:43,400] INFO - scETM.logging_utils: UnsupervisedTrainer.__init__(scETM(
  (q_delta): Sequential(
    (0): Linear(in_features=5634, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.1, inplace=False)
  )
  (mu_q_delta): Linear(in_features=128, out_features=50, bias=True)
  (logsigma_q_delta): Linear(in_features=128, out_features=50, bias=True)
  (rho_trainable_emb): PartlyTrainableParameter2D(height=400, fixed=0, trainable=5634)
), AnnData object with n_obs × n_vars = 1560000 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types', train_instance_name = scETM_training, ckpt_dir = ./scetm_results)
[2025-09-30 12:00:43,401] WARNING - scETM: Reinitializing... The file handler <FileHandler /home/krupavardhan4869@gmail.com/scetm_resu

[2025-09-30 12:00:45,057] INFO - scETM.trainers.UnsupervisedTrainer: ==========Epoch 0==========
[2025-09-30 12:00:45,058] INFO - scETM.trainers.UnsupervisedTrainer: pmem(rss=38595633152, vms=51112030208, shared=624832512, text=3207168, lib=0, data=39901003776, dirty=0)
[2025-09-30 12:00:45,059] INFO - scETM.trainers.UnsupervisedTrainer: lr          :        0.005
[2025-09-30 12:00:45,061] INFO - scETM.trainers.UnsupervisedTrainer: kl_weight   :            0
[2025-09-30 12:00:45,062] INFO - scETM.trainers.trainer_utils: loss        :      17.66
[2025-09-30 12:00:45,063] INFO - scETM.trainers.trainer_utils: nll         :      17.66
[2025-09-30 12:00:45,064] INFO - scETM.trainers.trainer_utils: kl_delta    :     0.5467
[2025-09-30 12:00:45,065] INFO - scETM.trainers.trainer_utils: max_norm    :       11.2
[2025-09-30 12:00:45,065] INFO - scETM.trainers.UnsupervisedTrainer: ==========End of evaluation==========


KeyboardInterrupt: 

In [ ]:
embeddings = combined_data.obsm['X_scetm'] 
np.save("combined_scETM_embeddings.npy", embeddings)

In [ ]:
combined_data.write_h5ad("combined_data_with_scETM.h5ad")

In [265]:
combined_data_scETM = sc.read_h5ad("combined_data_with_scETM.h5ad")

In [266]:
combined_data_scETM

AnnData object with n_obs × n_vars = 15600 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types'
    obsm: 'X_scetm', 'theta'

In [ ]:
combined_data_scETM

AnnData object with n_obs × n_vars = 15600 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types'
    obsm: 'X_scetm', 'theta'

Scaranoma

In [167]:
combined_data.obsm['X_scanorama']=run_scanorama_integration(combined_data,batch_key='batch')

Found 5634 genes among all datasets


[[0.         0.41474359]
 [0.         0.        ]]
Processing datasets (0, 1)


In [ ]:
combined_data_scar=combined_data[:780000].copy()

In [172]:
combined_data_scar

AnnData object with n_obs × n_vars = 7800 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types'
    obsm: 'theta', 'X_scetm', 'X_scanorama'

In [173]:
kmeans_clustering(combined_data_scar,embeddings='X_scanorama')

array([0, 4, 2, ..., 0, 8, 2], shape=(7800,), dtype=int32)

In [174]:
pred_labels = combined_data_scar.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.2359
ARI score: 0.1112


In [182]:
embeddings = combined_data_scETM.obsm['X_scetm'] 

In [188]:
combined_data_scETM = an.AnnData(
    X = embeddings,
    obs = combined_data_scETM.obs.copy()
)

In [189]:
combined_data_scETM.obsm['X_scanorama']=run_scanorama_integration(combined_data_scETM,batch_key='batch')

Found 50 genes among all datasets


[[0.         0.39512821]
 [0.         0.        ]]
Processing datasets (0, 1)


In [ ]:
combined_data_scETM_cpy = combined_data_scETM[:780000].copy()

In [192]:
kmeans_clustering(combined_data_scETM_cpy,embeddings='X_scanorama')
pred_labels = combined_data_scETM_cpy.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.3052
ARI score: 0.1898


In [193]:
combined_data_scETM

AnnData object with n_obs × n_vars = 15600 × 50
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types', 'cluster'
    obsm: 'X_scanorama'

Harmony

In [204]:
combined_data

AnnData object with n_obs × n_vars = 15600 × 5634
    obs: 'cell_type_level1', 'total_counts', 'batch', 'batch_indices', 'cell_types'
    obsm: 'theta', 'X_scetm', 'X_scanorama'

In [205]:
sc.pp.normalize_total(combined_data, target_sum=1e4)
sc.pp.log1p(combined_data)
sc.tl.pca(combined_data, n_comps=50)

normalizing counts per cell
    finished (0:00:00)
computing PCA
    with n_comps=50
    finished (0:00:04)


In [206]:
run_harmony_integration(combined_data,batch_key='batch')

2025-09-30 11:16:06,258 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-09-30 11:16:08,246 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-09-30 11:16:08,304 - harmonypy - INFO - Iteration 1 of 20
2025-09-30 11:16:10,047 - harmonypy - INFO - Iteration 2 of 20
2025-09-30 11:16:11,732 - harmonypy - INFO - Converged after 2 iterations


array([[-13.5828905 , -16.081587  ,  17.157171  , ...,  -0.35120115,
          0.77885234,  -1.2786455 ],
       [ -9.982015  , -15.115486  ,   3.827261  , ...,   0.5709206 ,
         -1.0253111 ,   2.8714712 ],
       [ -5.267168  ,  -0.12148676,  -7.489673  , ...,   0.13863768,
         -1.5326997 ,   0.9572785 ],
       ...,
       [ 10.409014  ,   2.8244236 ,   0.48581484, ...,   0.18023081,
         -3.2477748 ,  -1.4370123 ],
       [  9.306589  ,   3.5768409 ,   2.7303076 , ...,  -1.112727  ,
          0.56769794,  -0.595323  ],
       [  8.72492   ,  -0.3028943 ,   0.95824987, ...,  -1.4092509 ,
         -0.12502375,   0.5472048 ]], shape=(15600, 50), dtype=float32)

In [ ]:
combined_data_cpy = combined_data[:780000].copy()

In [208]:
kmeans_clustering(combined_data_cpy,embeddings='X_pca_harmony')
pred_labels = combined_data_cpy.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.3054
ARI score: 0.1657


In [197]:
combined_data_scETM.obsm['X_pca'] = combined_data_scETM.X

In [198]:
run_harmony_integration(combined_data_scETM,batch_key='batch')

2025-09-30 11:13:53,948 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-09-30 11:13:55,868 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-09-30 11:13:55,904 - harmonypy - INFO - Iteration 1 of 20
2025-09-30 11:13:56,563 - harmonypy - INFO - Iteration 2 of 20
2025-09-30 11:13:57,150 - harmonypy - INFO - Iteration 3 of 20
2025-09-30 11:13:57,795 - harmonypy - INFO - Converged after 3 iterations


array([[4.8452086, 4.6481237, 3.3051386, ..., 4.3533955, 5.692965 ,
        6.126924 ],
       [5.70048  , 4.6613264, 3.25984  , ..., 3.931786 , 5.158673 ,
        6.019462 ],
       [5.391483 , 5.3380737, 5.037695 , ..., 6.0171003, 5.3612766,
        5.2280474],
       ...,
       [4.87864  , 5.3057323, 5.3920875, ..., 5.6308   , 5.4214277,
        5.345077 ],
       [5.0734076, 5.199464 , 4.8353024, ..., 5.542753 , 5.493946 ,
        5.692694 ],
       [4.67742  , 5.2802405, 5.1085696, ..., 5.7821183, 5.3567643,
        5.3740478]], shape=(15600, 50), dtype=float32)

In [ ]:
combined_data_scETM_cpy = combined_data_scETM[:780000].copy()

In [203]:
kmeans_clustering(combined_data_scETM_cpy,embeddings='X_pca_harmony')
pred_labels = combined_data_scETM_cpy.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.3023
ARI score: 0.1692


scVI

In [209]:
run_scvi_integration(combined_data,batch_key='batch')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training scVI using **GPU**
Epoch 200/200: 100%|██████████| 200/200 [04:34<00:00,  1.36s/it, v_num=1, train_loss_step=1.82e+3, train_loss_epoch=1.83e+3]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [04:34<00:00,  1.37s/it, v_num=1, train_loss_step=1.82e+3, train_loss_epoch=1.83e+3]


array([[-0.23286414, -0.9775331 ,  1.2699673 , ...,  0.7646308 ,
        -2.5673156 , -0.39752138],
       [-1.4791625 , -1.2430732 , -0.43658075, ..., -1.5147213 ,
        -2.576653  , -1.4904853 ],
       [-0.94156194,  1.3580285 ,  1.780855  , ..., -1.9489998 ,
        -0.5865159 , -0.5961553 ],
       ...,
       [-0.12312388, -1.0017643 ,  0.94336164, ...,  0.33290112,
        -0.06211281,  0.991602  ],
       [ 0.48810554, -1.7060567 ,  0.47398615, ...,  1.1032797 ,
        -0.9311762 ,  0.7591182 ],
       [-0.02789354,  1.9398571 ,  1.2900127 , ...,  0.84346354,
        -1.671272  ,  0.40718132]], shape=(15600, 10), dtype=float32)

In [ ]:
combined_data_scVI = combined_data[:780000].copy()

In [211]:
kmeans_clustering(combined_data_scVI,embeddings='X_scvi')
pred_labels = combined_data_scVI.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.3199
ARI score: 0.2228


In [212]:
run_scvi_integration(combined_data_scETM,batch_key='batch')


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training scVI using **GPU**
Epoch 200/200: 100%|██████████| 200/200 [03:46<00:00,  1.14s/it, v_num=1, train_loss_step=88.6, train_loss_epoch=88.5]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [03:46<00:00,  1.13s/it, v_num=1, train_loss_step=88.6, train_loss_epoch=88.5]


array([[-1.0600470e-02, -1.5090002e+00,  4.9722265e-03, ...,
        -9.5970361e-03,  2.7536521e-02,  6.9184881e-03],
       [-7.6795788e-03, -1.3636035e+00,  2.3803697e-03, ...,
        -1.4830226e-02,  2.7020507e-02,  1.6655218e-02],
       [ 1.7264793e-02,  6.2441093e-01,  2.3105708e-03, ...,
        -3.0070446e-02,  1.0849067e-02,  2.2914773e-02],
       ...,
       [-1.0276812e-03,  7.0773214e-02, -1.4322237e-02, ...,
         4.5908275e-03, -5.5128923e-03, -3.0827080e-04],
       [-5.4509565e-03, -9.2823759e-02, -1.1207301e-02, ...,
        -7.6732243e-04,  1.6956569e-03, -2.3093575e-03],
       [ 8.2832784e-04,  8.5598156e-02, -1.6352903e-02, ...,
         2.4685632e-03, -1.1240714e-04,  1.2590409e-03]],
      shape=(15600, 10), dtype=float32)

In [ ]:
combined_data_scVI_cpy = combined_data_scVI[:780000].copy()

In [214]:
kmeans_clustering(combined_data_scVI_cpy,embeddings='X_scvi')
pred_labels = combined_data_scVI_cpy.obs['cluster'].values

nmi_score = normalized_mutual_info_score(true_labels, pred_labels)
ari_score = adjusted_rand_score(true_labels, pred_labels)

print(f"NMI score: {nmi_score:.4f}")
print(f"ARI score: {ari_score:.4f}")

NMI score: 0.3199
ARI score: 0.2228
